# Data Check
Verifies regular season and postseason parquets are complete and sane.
Run top-to-bottom after `uv run fetch.py` completes.

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

DATA_DIR = Path('data')

REG_YEARS  = list(range(2021, 2027))
POST_YEARS = list(range(2021, 2026))
POST_TYPES = ['F', 'D', 'L', 'W']  # Wild Card, Division Series, LCS, World Series

PALETTE = px.colors.qualitative.Plotly

def ok(msg):  print(f'  ✓  {msg}')
def warn(msg): print(f'  ⚠  {msg}')
def fail(msg): print(f'  ✗  {msg}')

## 1. Regular season pitches

In [26]:
print('Regular season pitches')
p = DATA_DIR / 'pitches.parquet'
if not p.exists():
    fail('pitches.parquet not found — run fetch.py')
else:
    pitches = pd.read_parquet(p)
    ok(f'{len(pitches):,} rows  |  {pitches.shape[1]} cols')
    ok(f'Date range: {pitches["game_date"].min().date()} → {pitches["game_date"].max().date()}')

    years_present = sorted(pitches['game_date'].dt.year.unique())
    missing_years = [y for y in REG_YEARS if y not in years_present]
    if missing_years:
        fail(f'Missing regular season years: {missing_years}')
    else:
        ok(f'All expected years present: {years_present}')

    by_year = pitches.groupby(pitches['game_date'].dt.year).size().reset_index(name='pitches')
    by_year.columns = ['year', 'pitches']
    print()
    print(by_year.to_string(index=False))

    fig = px.bar(by_year, x='year', y='pitches', title='Regular season pitches per year',
                 color_discrete_sequence=[PALETTE[0]])
    fig.update_layout(height=300, margin=dict(t=50, b=20))
    fig.show()

Regular season pitches
  ✓  1,441,877 rows  |  9 cols
  ✓  Date range: 2021-04-01 → 2026-08-17
  ✓  All expected years present: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

 year  pitches
 2021   251939
 2022   250039
 2023   250620
 2024   248937
 2025   248709
 2026   191633


## 2. Regular season game logs

In [27]:
print('Regular season game logs')
p = DATA_DIR / 'gamelogs.parquet'
if not p.exists():
    fail('gamelogs.parquet not found — run fetch.py')
else:
    gamelogs = pd.read_parquet(p)
    ok(f'{len(gamelogs):,} rows  |  {gamelogs.shape[1]} cols')
    ok(f'Date range: {gamelogs["game_date"].min().date()} → {gamelogs["game_date"].max().date()}')
    ok(f'Unique pitchers: {gamelogs["pitcher_id"].nunique():,}')

    years_present = sorted(gamelogs['game_date'].dt.year.unique())
    missing_years = [y for y in REG_YEARS if y not in years_present]
    if missing_years:
        fail(f'Missing regular season years: {missing_years}')
    else:
        ok(f'All expected years present: {years_present}')

    by_year = gamelogs.groupby(gamelogs['game_date'].dt.year).agg(
        rows=('pitcher_id', 'count'),
        pitchers=('pitcher_id', 'nunique'),
    ).reset_index()
    by_year.columns = ['year', 'rows', 'pitchers']
    print()
    print(by_year.to_string(index=False))

    # ERA sanity
    gl_ip = gamelogs[gamelogs['IP'] > 0].copy()
    sample_era = (gl_ip.groupby(gl_ip['game_date'].dt.year)
                  .apply(lambda g: g['ER'].sum() * 9 / g['IP'].sum())
                  .reset_index(name='lg_era'))
    sample_era.columns = ['year', 'lg_era']
    print()
    print('League ERA by year (should be ~4.0–4.6):')
    print(sample_era.to_string(index=False))
    for _, row in sample_era.iterrows():
        if not 3.0 <= row['lg_era'] <= 6.0:
            warn(f'{int(row["year"])} league ERA = {row["lg_era"]:.2f} — looks off')
        else:
            ok(f'{int(row["year"])} ERA = {row["lg_era"]:.2f}')

Regular season game logs
  ✗  gamelogs.parquet not found — run fetch.py


## 3. Postseason pitches

In [28]:
print('Postseason pitches')
p = DATA_DIR / 'pitches_post.parquet'
if not p.exists():
    fail('pitches_post.parquet not found — run fetch.py')
else:
    pitches_post = pd.read_parquet(p)
    ok(f'{len(pitches_post):,} rows  |  {pitches_post.shape[1]} cols')
    ok(f'Date range: {pitches_post["game_date"].min().date()} → {pitches_post["game_date"].max().date()}')
    ok(f'Unique pitchers: {pitches_post["pitcher"].nunique():,}')

    years_present = sorted(pitches_post['game_date'].dt.year.unique())
    missing_years = [y for y in POST_YEARS if y not in years_present]
    if missing_years:
        fail(f'Missing postseason years: {missing_years}')
    else:
        ok(f'All expected postseason years present: {years_present}')

    by_year = pitches_post.groupby(pitches_post['game_date'].dt.year).agg(
        pitches=('game_pk', 'count'),
        games=('game_pk', 'nunique'),
        pitchers=('pitcher', 'nunique'),
    ).reset_index()
    by_year.columns = ['year', 'pitches', 'games', 'pitchers']
    print()
    print(by_year.to_string(index=False))

    for yr in POST_YEARS:
        n = (pitches_post['game_date'].dt.year == yr).sum()
        if n < 2000:
            warn(f'{yr} postseason: only {n:,} pitches — may be incomplete')
        else:
            ok(f'{yr} postseason: {n:,} pitches')

    fig = px.bar(by_year, x='year', y='pitches', title='Postseason pitches per year',
                 color_discrete_sequence=[PALETTE[1]])
    fig.update_layout(height=300, margin=dict(t=50, b=20))
    fig.show()

Postseason pitches
  ✗  pitches_post.parquet not found — run fetch.py


## 4. Postseason game logs

In [29]:
print('Postseason game logs')
p = DATA_DIR / 'gamelogs_post.parquet'
if not p.exists():
    fail('gamelogs_post.parquet not found — run fetch.py')
else:
    gamelogs_post = pd.read_parquet(p)
    ok(f'{len(gamelogs_post):,} rows  |  {gamelogs_post.shape[1]} cols')
    ok(f'Unique pitchers: {gamelogs_post["pitcher_id"].nunique():,}')

    years_present = sorted(gamelogs_post['game_date'].dt.year.unique())
    missing_years = [y for y in POST_YEARS if y not in years_present]
    if missing_years:
        fail(f'Missing postseason years in game logs: {missing_years}')
    else:
        ok(f'All expected postseason years present: {years_present}')

    # Game type coverage
    if 'game_type' in gamelogs_post.columns:
        types_present = sorted(gamelogs_post['game_type'].unique())
        missing_types = [t for t in POST_TYPES if t not in types_present]
        if missing_types:
            warn(f'Missing game types: {missing_types}')
        else:
            ok(f'All game types present: {types_present}  (F=Wild Card, D=DS, L=LCS, W=WS)')

        by_year_type = (
            gamelogs_post[gamelogs_post['IP'] > 0]
            .groupby([gamelogs_post['game_date'].dt.year, 'game_type'])
            .agg(appearances=('pitcher_id', 'count'), pitchers=('pitcher_id', 'nunique'))
            .reset_index()
        )
        by_year_type.columns = ['year', 'game_type', 'appearances', 'pitchers']
        print()
        print(by_year_type.to_string(index=False))

        fig = px.bar(by_year_type, x='year', y='appearances', color='game_type',
                     barmode='stack', title='Postseason appearances by year and round',
                     category_orders={'game_type': POST_TYPES})
        fig.update_layout(height=350, margin=dict(t=50, b=20))
        fig.show()

    # ERA sanity per round
    print()
    print('Postseason ERA by round (higher variance expected, ~4–5 typical):')
    gl_ip = gamelogs_post[gamelogs_post['IP'] > 0]
    for gt in POST_TYPES:
        sub = gl_ip[gl_ip['game_type'] == gt]
        if len(sub) == 0:
            warn(f'{gt}: no data')
            continue
        era = sub['ER'].sum() * 9 / sub['IP'].sum()
        ok(f'{gt}: ERA {era:.2f}  ({len(sub):,} appearances)')

Postseason game logs
  ✗  gamelogs_post.parquet not found — run fetch.py


## 5. Pitcher ID overlap — postseason vs regular season

In [30]:
print('Pitcher ID overlap')

reg_ids  = set(gamelogs['pitcher_id'].unique())
post_ids = set(gamelogs_post['pitcher_id'].unique())

in_both    = post_ids & reg_ids
post_only  = post_ids - reg_ids

ok(f'Regular season pitchers: {len(reg_ids):,}')
ok(f'Postseason pitchers:     {len(post_ids):,}')
ok(f'In both:                 {len(in_both):,}  ({100*len(in_both)/len(post_ids):.1f}% of postseason)')

if post_only:
    warn(f'{len(post_only)} postseason pitchers have no regular season game log — may be pre-2021 callups or data gaps')
    print(f'  Sample IDs: {sorted(post_only)[:10]}')
else:
    ok('All postseason pitchers have regular season data')

# Postseason pitchers per year
print()
print('Postseason pitchers per year:')
for yr in POST_YEARS:
    n = gamelogs_post[gamelogs_post['game_date'].dt.year == yr]['pitcher_id'].nunique()
    ok(f'{yr}: {n} pitchers')

Pitcher ID overlap


NameError: name 'gamelogs_post' is not defined

## 6. Summary

In [ ]:
print('=== Data summary ===')
rows = [
    ('pitches.parquet',      f'{len(pitches):,}',      f'{pitches["game_date"].min().date()} → {pitches["game_date"].max().date()}'),
    ('gamelogs.parquet',     f'{len(gamelogs):,}',     f'{gamelogs["game_date"].min().date()} → {gamelogs["game_date"].max().date()}'),
    ('pitches_post.parquet', f'{len(pitches_post):,}', f'{pitches_post["game_date"].min().date()} → {pitches_post["game_date"].max().date()}'),
    ('gamelogs_post.parquet',f'{len(gamelogs_post):,}',f'{gamelogs_post["game_date"].min().date()} → {gamelogs_post["game_date"].max().date()}'),
]
print(f'{"file":<26} {"rows":>12}  {"date range"}')
print('-' * 65)
for name, count, dates in rows:
    print(f'{name:<26} {count:>12}  {dates}')